# Kunskapskontroll 1 

## 1. Inledning och kontext

Detta dataset representerar en simulering och statististisk sammaställning av de 48 landslag som förväntas delta i **FIFA World Cup 2026**

**Om datan:**
- **Källor:** Det används två filer från kaggle: fifa_history.csv ( historisk statistik från tidigare vm) samt fifa_2026.csv (statisitk simulering av de 48 lagen inför vm 2026)
- **Population:** Varje rad representerar ett specifikt landslag. 
- **Variabler:** Datasetet innehåller 24 kolumner med allt från historiska meriter (tidigare titlar) till nuvarande form (mål och vinster de senaste 4 åren)
- **Syfte:** Att undersöka faktorer som påverkar ett lags förväntade framgång inför det utökade VM-formatet
- **Begränsningar:*Eftersom jag slår ihop två dataset där vi har historisk data och framtida data. Detta gör att viss data kommer inte att finnas till exempel på winner för 2026. jag väljer att radera dessa tomma rader automatiskt. Detta för att inte medvetet blanda ihop faktiska resultat med framtidsprogrnoser utifrån missvisande sätt.  



In [ ]:
# importera de olika paketen. Vi använder oss av matplotlib och seabord för att skapa visuella diagram 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### hämta in dataseten som jag ska arbeta med och se hur stora de är

In [ ]:
# Ladda in alla dataseten som vi arbetar med
df_history = pd.read_csv("data/fifa_history.csv")
df_2026 = pd.read_csv("data/fifa_2026.csv")
df_players = pd.read_csv(f"data/players.csv")
df_values = pd.read_csv(f"data/transfermarkt.csv")

# Inspektera historisk data för tidigare vm
print("history table")
print(f"Historik (Train): {df_history.shape[0]} rader, {df_history.shape[1]} kolumner")
display(df_history.head(3))

print(" ")

# Inspektera framtida data
print("wc 2026 table")
print(f"VM 2026 (Test): {df_2026.shape[0]} rader, {df_2026.shape[1]} kolumner")
display(df_2026.head(3))

print(" ")

# Inspektera spelardata
print("players table")
print(f"Spelardata: {df_players.shape[0]} rader, {df_players.shape[1]} kolumner")
display(df_players.head(3))

print(" ")

# Inspektera marknadsvärdesdata
print("transfermarkt table")
print(f"Marknadsvärde: {df_values.shape[0]} rader, {df_values.shape[1]} kolumner")
display(df_values.head(3))

### Sammanslagningen
här slår vi ihop historiska och framtida dataseten till en. 

In [ ]:
# 1. (Valfritt men smart) Ge dem en "etikett" så vi vet vad som är vad sen
df_history["status"] = "Historik"
df_2026["status"] = "2026"

# 2. Slå ihop dem 
# ignore_index=True gör att vi får en ny snygg numrering från 0 till slutet
df = pd.concat([df_history, df_2026], ignore_index=True)

# 3. Kontrollera resultatet
print(f"Nu har vi slagit ihpa de två dataseten och då får vi totalt antal rader: {df.shape[0]}") 

print (" ")
print(" här ser vi slutet av datasetet där data från prediktion är.")
display(df.tail())  
df.isnull().sum() # kollar antalet icke siffer värden 

Nu har vi slagit ihpa de två dataseten och då får vi totalt antal rader: 240
 
 här ser vi slutet av datasetet där data från prediktion är.


version                             0
team                                0
continent                           0
is_host                             0
goals_scored_last_4y                0
goals_received_last_4y              0
wins_last_4y                        0
losses_last_4y                      0
draws_last_4y                       0
world_cup_titles_before             0
squad_total_market_value_eur       32
fifa_rank_pre_tournament            0
fifa_points_pre_tournament          0
squad_avg_age                       0
world_cup_participations_before     0
groups_passed_before                0
round16_before                      0
quarterfinals_before                0
semifinals_before                   0
finals_before                       0
winner                             48
finalist                           48
semi_finalist                      48
quarter_finalist                   48
status                              0
dtype: int64

### data integration : 
aggregering av spelarvärden och skapa en enhetlig dataset

In [ ]:
# 1. En funktion som fattar skillnaden på 'm' och 'k'
def clean_market_value(val):
    if pd.isna(val) or val == "":
        return 0.0

    val = str(val).lower().replace("€", "").strip()

    if "m" in val:
        # '180.00m' -> 180.0
        return float(val.replace("m", ""))
    elif "k" in val:
        # '800k' -> 0.8 (vi gör om tusen till miljoner för att matcha)
        return float(val.replace("k", "")) / 1000

    try:
        return float(val)
    except:
        return 0.0


# 2. Kör funktionen på MarketValue
df_players["MarketValue"] = df_players["MarketValue"].apply(clean_market_value)

# 3. Nu kan du gruppera precis som förut!
df_economy = (
    df_players.groupby("Nationality")
    .agg(
        {
            "MarketValue": "sum",
            "Age": "mean",
            "PlayerName": "count",
        }
    )
    .reset_index()
)

# 4. sparar en kopia som vi döper med
df_economy.columns = ["team_name", "total_market_value", "avg_age", "squad_size"]
df_before_merge = df.copy()

# 5. Merge
df_final = pd.merge(df, df_economy, left_on="team", right_on="team_name", how="left")

# 6. Fyll 0 för de som saknas
df_final["total_market_value"] = df_final["total_market_value"].fillna(0)

print("Tvätten klar! Nu är alla 'k' och 'm' omgjorda till siffror (i miljoner).")

## 3. Datatvätt 
ta bort data(NaN) som vi inte behöver och som vi inte vill hantera för att inte få felaktiga resultat

Efter sammanslagningen ser vi att kolumnerna winner, finalist, semi_finalist och quarter_finalist har många saknade värden.

Motivering: Dessa NaN beror inte på felaktig datainsamling, utan på att 2026-turneringen ännu inte har spelats. För att kunna använda dessa kolumner i numeriska analyser väljer jag att imputera (fylla) dessa värden med 0. Detta representerar att lagen i projektionen för 2026 än så länge har noll vinster eller finalplatser i det kommande mästerskapet.

In [ ]:
# Lista på de kolumner som rör slutplaceringar
cols_to_fix = ["winner", "finalist", "semi_finalist", "quarter_finalist"]

# Fyll alla NaN i dessa kolumner med 0
df[cols_to_fix] = df[cols_to_fix].fillna(0)

# Verifiera att de saknade värdena är borta i just dessa kolumner
print("Antal saknade värden efter tvätt:")
print(df[cols_to_fix].isnull().sum())

## datatyper

Här vill vi verifiera så att de olika kolumnerna har rätt datatyper. Detta är för att vi ska kunna arbeta med dem på rätt sätt. 
Här listar jag alla kolumneran och här ser vi också att det inte finns några objekt. det hade medfört problem när vi ska räkna ut medelvärden eller skapa grafer

In [ ]:
# Kolla vilka typer kolumnerna har
print(df.dtypes)

### Gör om resultat-kolumnerna till heltal
verifierar så att det stämmer undertill med

In [ ]:
cols_to_int = ["winner", "finalist", "semi_finalist", "quarter_finalist"]
df[cols_to_int] = df[cols_to_int].astype(int)

# Kontrollera att det blev rätt
print(df[cols_to_int].dtypes)

### Kolla så att samma kontinent inte har flera olika namn
Så det ska vara Unika kontinenter i datasetet

In [ ]:
print("Unika kontinenter i datasetet:")
print(df["continent"].unique())

### Tar bort eventuella osynliga mellanslag i början eller slutet av namnen


In [ ]:
df["team"] = df["team"].str.strip()

In [ ]:
# Skapa en kolumn för "Målskillnad" de senaste 4 åren
df["goal_diff_4y"] = df["goals_scored_last_4y"] - df["goals_received_last_4y"]

# Skapa en kolumn för "Win Rate" (vinstprocent)
df["win_rate"] = np.where(
    (df["wins_last_4y"] + df["losses_last_4y"] + df["draws_last_4y"]) > 0,
    df["wins_last_4y"]
    / (df["wins_last_4y"] + df["losses_last_4y"] + df["draws_last_4y"]),
    0,
)

In [ ]:
# Jämför genomsnittligt marknadsvärde per kontinent
continent_analysis = (
    df.groupby("continent")["squad_total_market_value_eur"]
    .mean()
    .sort_values(ascending=False)
)
print(continent_analysis)

In [ ]:
# Kolla korrelationen mellan marknadsvärde och vinster
correlation = df["squad_total_market_value_eur"].corr(df["wins_last_4y"])
print(f"Sambandet mellan värde och vinster är: {correlation:.2f}")

In [ ]:
# Vi sorterar först efter kontinent och sen efter antal finaler (fallande)
top_finalists = df.sort_values(["continent", "finals_before"], ascending=[True, False])

# Nu behåller vi bara den första raden för varje kontinent (alltså den med flest finaler)
top_finalists_per_continent = top_finalists.drop_duplicates(subset="continent")

# Vi väljer ut de kolumner som är intressanta att visa
result = top_finalists_per_continent[["continent", "team", "finals_before"]]

print("Lag med flest finalplatser per kontinent:")
display(result)

Analys av finalplatser:
"Resultatet bekräftar en historisk dominans från Europa och Sydamerika. Det är intressant att notera att Tyskland (8) har fler finalplatser än Brasilien (6), trots att Brasilien har fler titlar totalt. För övriga kontinenter ser vi att ingen nation ännu nått en final, vilket belyser den stora utmaningen för dessa lag inför det utökade mästerskapet 2026."

In [ ]:
# Visa både finaler och titlar för topplagen
result_extended = top_finalists_per_continent[
    ["continent", "team", "finals_before", "world_cup_titles_before"]
]
display(result_extended)

In [ ]:
# Summera alla finaler per kontinent
continent_total_finals = (
    df.groupby("continent")["finals_before"].sum().sort_values(ascending=False)
)

print("Totala antalet finalplatser per kontinent (historiskt):")
print(continent_total_finals)

In [ ]:
# Hitta länder som inte fick någon matchning i spelar-datan
missing_data = df_final[df_final["total_market_value"].isnull()]["team"].unique()

if len(missing_data) > 0:
    print(f"Varning! Dessa länder saknar marknadsvärde: {missing_data}")
else:
    print("Snyggt! Alla länder matchades perfekt.")

In [ ]:
# Nu fungerar nlargest eftersom det är siffror!
print("Topp 5 dyraste trupperna:")
display(df_final.nlargest(5, "total_market_value")[["team", "total_market_value"]])

print("\n5 billigaste trupperna:")
display(df_final.nsmallest(5, "total_market_value")[["team", "total_market_value"]])

In [ ]:
print(f"Rader innan merge: {df_before_merge.shape[0]}")
print(f"Rader efter merge: {df_final.shape[0]}")

In [74]:
# Använd df_final här!
winner_value_check = df_final.groupby("world_cup_titles_before")[
    "total_market_value"
].mean()
print("Snittvärde baserat på antal VM-titlar:")
print(winner_value_check)

Snittvärde baserat på antal VM-titlar:
world_cup_titles_before
0     173.489950
1    1240.900000
2     700.900000
3     591.233333
4     675.040000
5    1359.200000
Name: total_market_value, dtype: float64


## Diagram

### 1. vilka lag har har den dyraste truppen inför vm 2026 

vi börjar med att sortera ut de 10 dyraste lagen utifrån marknads värdet och skriver ut det så att vi kan verifiera så att datan ser ok ut

In [ ]:

top_10_expensive = df_final[df_final["status"] == "2026"].nlargest(
    10, "total_market_value"
)

print(top_10_expensive)

#plt.figure(figsize=(12, 6))
#sns.barplot(data=top_10_expensive, x="total_market_value", y="team", palette="viridis")

#plt.title("Top 10 dyraste landslagen inför VM 2026", fontsize=15)
#plt.xlabel("Totalt marknadsvärde (Miljoner €)", fontsize=12)
#plt.ylabel("Landslag", fontsize=12)
#plt.grid(axis="x", linestyle="--", alpha=0.7)

#plt.tight_layout()
#plt.savefig("top_10_market_value.png")

     version         team      continent  is_host  goals_scored_last_4y  \
192     2026       France         Europe        0                    85   
197     2026       Brazil  South America        0                    58   
195     2026      England         Europe        0                    82   
193     2026        Spain         Europe        0                   104   
194     2026    Argentina  South America        0                    80   
196     2026     Portugal         Europe        0                    98   
198     2026  Netherlands         Europe        0                    92   
200     2026      Belgium         Europe        0                    80   
201     2026      Germany         Europe        0                    80   
199     2026      Morocco         Africa        0                   100   

     goals_received_last_4y  wins_last_4y  losses_last_4y  draws_last_4y  \
192                      32            25               6              7   
197                   

In [85]:
top_10_expensive = df_final[df_final["status"] == "2026"].nlargest(10, "total_market_value")

print(top_10_expensive)

     version         team      continent  is_host  goals_scored_last_4y  \
192     2026       France         Europe        0                    85   
197     2026       Brazil  South America        0                    58   
195     2026      England         Europe        0                    82   
193     2026        Spain         Europe        0                   104   
194     2026    Argentina  South America        0                    80   
196     2026     Portugal         Europe        0                    98   
198     2026  Netherlands         Europe        0                    92   
200     2026      Belgium         Europe        0                    80   
201     2026      Germany         Europe        0                    80   
199     2026      Morocco         Africa        0                   100   

     goals_received_last_4y  wins_last_4y  losses_last_4y  draws_last_4y  \
192                      32            25               6              7   
197                   